In [ ]:
from model_wrapper import *
from sklearn.externals.array_api_extra.testing import override

In [ ]:
import tensorflow as tf
import numpy as np
import logging

tf.get_logger().setLevel('ERROR')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

def load_data_wrapper(model):
    def load_data(folder_path, label):
        def process_instance(path_tensor):
            actual_path = path_tensor.numpy().decode('utf-8')
            x = get_training_instance(actual_path)
            x = np.transpose(x, (1, 2, 0))
            return x.astype(np.float32)

        image_data = tf.py_function(
            func=process_instance,
            inp=[folder_path],
            Tout=tf.float32
        )

        image_data.set_shape([IMG_HEIGHT, IMG_WIDTH, MIN_IMG_COUNT])

        return image_data, np.reshape(label[:, model.target_index], shape=(label.shape[0], 1))
    return load_data
class CNNModel(MultiLabelModel):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, no_valid=False):
        blue_print = tf.keras.models.Sequential([
            tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, MIN_IMG_COUNT)),
            tf.keras.layers.Conv2D(18, 8, activation="relu"),
            tf.keras.layers.MaxPool2D(pool_size=4),
            tf.keras.layers.Conv2D(12, 5, activation="relu"),
            tf.keras.layers.MaxPool2D(pool_size=3),
            tf.keras.layers.Conv2D(6, 5, activation="relu"),
            tf.keras.layers.MaxPool2D(pool_size=3),
            tf.keras.layers.Conv2D(1, 3, activation="relu"),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(10, activation='tanh'),
            tf.keras.layers.Dense(len(target_columns), activation="sigmoid")
        ])
        blue_print.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                metrics=[tf.keras.metrics.AUC(multi_label=True, num_labels=len(target_columns), name='auc')])

        self.models = [blue_print.copy() for _ in range(len(target_columns))]
        self.history = []
        self.target_index = 0
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression, full_train=False, no_valid=no_valid)

    @override
    def batch_fit(self, training_folders: pd.Series, y_train: np.ndarray, validation_folders: pd.Series, y_validation: np.ndarray):
        train_dataset = tf.data.Dataset.from_tensor_slices((training_folders, y_train))
        train_dataset = train_dataset.map(load_data_wrapper(self), num_parallel_calls=tf.data.AUTOTUNE)
        train_dataset = train_dataset.batch(BATCH_SIZE)
        train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

        val_dataset = None
        if validation_folders is not None:
            val_dataset = tf.data.Dataset.from_tensor_slices((validation_folders, y_validation))
            val_dataset = val_dataset.map(load_data_wrapper(self), num_parallel_calls=tf.data.AUTOTUNE)
            val_dataset = val_dataset.batch(BATCH_SIZE)
            val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

        self.history.append(self.models[self.target_index].fit(
            train_dataset,
            epochs=5,
            validation_data=val_dataset,
            verbose=0
        ))

        self.target_index += 1
    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.transpose(x, (0, 2, 3, 1))
        pred_list = [tf.squeeze(m.predict(x, verbose=0)) for m in self.models]
        return np.hstack(pred_list)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.transpose(x, (1, 2, 0))
        x = np.expand_dims(x, axis=0)
        pred_ = [np.reshape(m.predict(x, verbose=0), shape=(0,)) for m in self.models]
        return np.array(pred_)

In [ ]:
ensemble: List[CNNModel] = []

for p in planes:
    for i in range(2):
        if p == "Axial" and i == 0:
            ensemble.append(CNNModel(p, i, i, no_valid=True))
        else:
            ensemble.append(CNNModel(p, i, i))

In [ ]:
i = 0
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()
target_name = target_columns[0]
print(f"{target_name} performance")
for p in planes:
    for j in range(2):
        df = pd.DataFrame(ensemble[i].history[target_columns.index(target_name)].history)
        df = df.rename(columns={"auc": "train_auc"})
        if "val_auc" in df.columns:
            df[["train_auc", "val_auc"]].plot(ax=axes[i])
        else:
            df[["train_auc"]].plot(ax=axes[i])

        axes[i].set_title(f"{p}: {j} - {j}")
        axes[i].set_ylim(0.3, 1.0)
        axes[i].axhline(y=0.5, color='red', linestyle='-')

        axes[i].set_xlabel("Epoch")
        axes[i].set_ylabel("AUC Score")
        axes[i].grid(True)
        i += 1

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(18, 10))
axes = axes.flatten()
plane = planes[0]
mri_mode = 0

print(f"Model: {plane} - {mri_mode}")
for i in range(len(target_columns)):
    model = None
    for m in ensemble:
        if m.anatomical_plane == plane and m.fat_suppression == mri_mode:
            model = m
            break

    df = pd.DataFrame(model.history[i].history)
    df = df.rename(columns={"auc": "train_auc"})
    if "val_auc" in df.columns:
        df[["train_auc", "val_auc"]].plot(ax=axes[i])
    else:
        df[["train_auc"]].plot(ax=axes[i])

    axes[i].set_title(f"{target_columns[i]} AUC Score")
    axes[i].set_ylim(0.3, 1.0)
    axes[i].axhline(y=0.5, color='red', linestyle='-')

    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("AUC Score")
    axes[i].grid(True)

In [ ]:
scores = MultiLabelModel.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

In [ ]:
scores = MultiLabelModel.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

In [ ]:
data = {"name": []}
for t in target_columns:
    data[t] = []

for m in ensemble:
    data["name"].append(f"{m.anatomical_plane}_{m.fat_suppression}_{m.fluid_sensitive}")
    for i in range(len(target_columns)):
        data[target_columns[i]].append(m.auc_scores[i])

data = pd.DataFrame(data)
data

In [ ]:
auc_scores = []
for t in target_columns:
    pos = data[t].argmax()
    auc_scores.append(data[t].iloc[pos])
    print(f"{t}: score -> {data[t].iloc[pos]:.3f}, model -> {data['name'].iloc[pos]}")
print(f"Average: {np.mean(auc_scores)}")